In [1]:
import sys, os

_proj_db = r'C:\Users\marrocol\AppData\Local\miniforge3\envs\mswe-gnn\Lib\site-packages\pyproj\proj_dir\share\proj'
os.environ.setdefault('PROJ_DATA', _proj_db)
os.environ.setdefault('PROJ_LIB',  _proj_db)

try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset
from utils.visualization import PlotRollout
from training.train import LightningTrainer

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')
print('Repo root:', REPO_ROOT)

Repo root: c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg


In [5]:
api = wandb.Api()

ENTITY   = "mmarrocolo-tu-delft"
PROJECT  = "mswe-gnn-ahr-sweep"
SWEEP_ID = "9hskioby"

sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

# best_run(order="val_loss") sorts DESCENDING → returns the WORST run.
# Sort manually to get the true minimum val_loss.
#runs = [r for r in sweep.runs if r.summary.get('val_loss') is not None]
#best_run = min(runs, key=lambda r: r.summary['val_loss'])
runs = [r for r in sweep.runs if r.summary.get('val_CSI_03') is not None]
best_run = max(runs, key=lambda r: r.summary['val_CSI_03'])

print(f"Best run : {best_run.name}  (id: {best_run.id})")
print(f"val_loss     : {best_run.summary.get('val_loss',    float('nan')):.4f}")
print(f"val_CSI_005  : {best_run.summary.get('val_CSI_005', float('nan')):.4f}")
print(f"val_CSI_03   : {best_run.summary.get('val_CSI_03',  float('nan')):.4f}")
print(f"\nHyperparameters:")
for k, v in sorted(best_run.config.items()):
    print(f"  {k}: {v}")

# Download the best checkpoint
ckpt_files = [f for f in best_run.files() if f.name.endswith('.ckpt')]
if ckpt_files:
    ckpt_path = ckpt_files[0].download(replace=True).name
    print(f"\nCheckpoint saved to: {ckpt_path}")
else:
    print("\nNo .ckpt file found — checkpoint may still be on hal8")

Best run : rollout5_hid32_selected_bestvalloss  (id: ffphz2b4)
val_loss     : 0.3215
val_CSI_005  : 0.6744
val_CSI_03   : 0.6772

Hyperparameters:
  K: [1, 1, 1, 5, 4, 3, 2]
  dataset_parameters: {'seed': 0, 'val_prcnt': 0, 'train_size': 1, 'temporal_res': 60, 'dataset_folder': 'database/datasets', 'test_dataset_name': 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart', 'train_dataset_name': 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart'}
  lr_info: {'T_max': 1000, 'gamma': 0.7, 'eta_min': 1e-06, 'scheduler': 'cosine', 'step_size': 10, 'weight_decay': 0, 'learning_rate': 0.0006868772974210106}
  lr_info.learning_rate: 0.0005177909168244793
  models: {'K': [1, 2, 2, 7, 5, 4, 3], 'seed': 666, 'with_WL': True, 'edge_mlp': True, 'normalize': True, 'mlp_layers': 3, 'model_type': 'MSGNN', 'hid_features': 64, 'with_gradient': True, 'gnn_activation': 'tanh', 'mlp_activation': 'prelu', 'learned_pooling': False, 'skip_connections': True, 'learned_residuals': True, 'with_filte